Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["GROK_API_KEY"]=os.getenv("GROK_API_KEY")

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

model=ChatGoogleGenerativeAI(model="gemini-3.5-flash")
response=model.invoke("Why do parrots talk?")
response

AIMessage(content=[{'type': 'text', 'text': 'Parrots talk due to a unique combination of **social instinct, specialized brain structure, and flexible vocal anatomy**. \n\nUnlike most animals, parrots are naturally wired to copy the sounds around them. Here is a breakdown of why and how they do it:\n\n---\n\n### 1. The Need to Fit In (Flock Dynamics)\nIn the wild, parrots are extremely social animals that live in tight-knit flocks. \n* **Wild Dialects:** Wild parrots use complex vocalizations to communicate, identify flock members, and warn against predators. Different flocks even have distinct "dialects."\n* **Humans as the "Flock":** When a parrot lives in a human household, it sees its human owners as its flock. Because parrots want to bond with their group, they naturally try to mimic the sounds their "human flock" makes to fit in and get attention.\n\n### 2. Specialized Brain Structure\nParrots are part of a rare group of animals known as **"vocal learners"** (which also includes h

In [6]:
for chunk in model.stream("Write me a 200 words paragraph on Artificial Intelligence"):
    print(chunk.text,end="|",flush=True)

Artificial Intelligence| (AI) represents one of the most transformative technological advancements of the modern era. At its core, AI refers to the| simulation of human intelligence in machines programmed to think, learn, adapt, and solve complex problems. Powered by sophisticated algorithms,| neural networks, and massive amounts of data, AI systems can process information at extraordinary speeds, identifying subtle patterns and making autonomous| decisions with remarkable precision. Today, AI is seamlessly integrated into daily life, driving everything from virtual smart assistants to personalized recommendation| systems on streaming platforms. Beyond consumer convenience, it is revolutionizing critical industries worldwide: in healthcare, AI assists in early| disease diagnosis and drug discovery; in finance, it detects fraudulent transactions; and in transportation, it powers autonomous vehicles. As| AI technology rapidly evolves, its potential to enhance human productivity and solve

Tool Execution Loops

In [8]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
   
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools=model.bind_tools([get_weather])

In [10]:
response=model_with_tools.invoke("What is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_385298': 'EtECCs4CAWkUfRP8wF0DaAMqmQLfqPb8KX2s7bML8cWoBeQPkL8GUXJDjdCu9+JIE9x5yAIrwuVbh3ecU1lP9MxdIsYfE9wVW/pjsu+GgDTMJqSJspIkEv19HItpB8P6ZczZxUlMUO0+ftO7rpqQw/DluwgfOeOxov0aJxd9SibNnRP6EWtBqyh34PYv5vfnQACMolcNQ7Ie+tAIFDzGaj7SgmEcKA2GbNUTdjZDh4Y7+/Qqlcbr8Dxt0TA4dGRSiurV3scM+28AOrGZrLyDmmsk2S/fZx9kSSiBQCLXcwpfZ1wwbaSz1OFKudY7GglQAHKNV3fFDpuGBWvpfhw3/pTHfdhuGbX69lo0DRP/uKlrMB7YdLNVn6t28ssrtTvSZ9bcygtTICKJw1wByY8i5KfQ0FihfPVtI/Hk2Uj/RfdGUDEGGu3KAfzkkkFqOFbW+q+8RA=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1ab8c-0b8f-713c-bd54-9d8cd3b7f9ec-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_385298', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 55, 'outpu

In [ ]:
### Tool execution

Tool Execution Loops

In [11]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Boston is sunny with a temperature of 25°C (77°F).


In [12]:

messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_396701': 'EpgCCpUCAWkUfRPIAEEQkfwGDaGo4LofiEqPIpzYziEsdrcttA4SeCGhryeGjNoFv7IZ/GeF+fKMyedo3vC5G2psKCmK2DJdiaez/+02V399EL1YAR+yOm84fVhspisd6nAwELtSKSCYXHUK5YqTIngwauOlQLy/Y9cSiv+lRHmKnwk/E4wgbDfQ5KXR0ZHKFIvb1l3wxPUQRgHIUN4V5cShOEQnN6dENzpRwxh4B5uCvxVfO7RPGGx1oCyh8IXaod7M644bVAZJG9+Y/EoLBw8tvQVPHBbWm9AWR+gNBsybYpREeP38h5YFe2fNzv7NDtMfPBx/HtXA12uaoywbpxoCYk8sa0p6dz4joO0ZKH0VrRZWBxgVyjPnrg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1ab8d-93aa-72ee-a255-4bfd027460e7-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_396701', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'outpu